In [1]:
# User_Table 
import os
import re
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# load
df = pd.read_csv(r'C:\Users\dell\Desktop\smart-home-product-analysis\datasets\raw_data\user_table.csv')

# quick look
print(df.shape)
print(df.duplicated().sum())
print(df['user_id'].nunique())
display(df.head(10))
display(df.isnull().sum().rename('missing').to_frame())

# check uuid format
uuid_pattern = r'^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$'
invalid = df[~df['user_id'].str.match(uuid_pattern, na=False)]
print(f"invalid user_ids: {len(invalid)}")
display(invalid.head())

# drop invalid ids and duplicates
df = df[df['user_id'].str.match(uuid_pattern, na=False)]
df = df.drop_duplicates(subset=['user_id'], keep='first')

# fix boolean columns
for col in ['has_purchase_last_year', 'has_purchase_last_qtr']:
    df[col] = df[col].map(lambda x: True if str(x).strip().upper() == 'TRUE' else False)

# segment users based on purchase history
conditions = [
    df['has_purchase_last_qtr'] == True,                                              # active
    (df['has_purchase_last_year'] == True) & (df['has_purchase_last_qtr'] == False)   # lapsed
]
choices = ['active', 'lapsed']
df['user_segment'] = np.select(conditions, choices, default='new_or_inactive')

print(df['user_segment'].value_counts())
print(df['user_segment'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

# final check
print(df.shape)
print(df['user_segment'].value_counts())
display(df.head(10))


# save
CLEANED = r'C:\Users\dell\Desktop\smart-home-product-analysis\datasets\cleaned_data'
os.makedirs(CLEANED, exist_ok=True)
df.to_csv(os.path.join(CLEANED, 'user_cleaned.csv'), index=False)
print("saved")

(102435, 3)
0
102435


,user_id,has_purchase_last_year,has_purchase_last_qtr
0,9d415f62-5d38-436d-a443-336b2b759568,False,False
1,a71251a4-f215-4046-85fb-51206780e6c8,True,True
2,a8e69945-e3f7-4458-8ea9-883062d75c71,False,False
3,d2280282-85c3-40fe-93b8-990479b73fb3,False,False
4,19755aa9-50fc-44f2-80c0-e2eb4ecb4160,True,True
5,b632f58c-f467-4363-9b87-b01c237df412,False,False
6,ff30683f-bd0b-4ada-9cc8-547049fdc62c,False,False
7,3cc6932b-7cc1-4852-b6a8-3348607240eb,False,False
8,2d268310-e659-49d4-82b1-b18f344587d1,True,True
9,5c135263-b6ee-4cec-b431-82e90a062983,False,False


,missing
user_id,0
has_purchase_last_year,0
has_purchase_last_qtr,0


invalid user_ids: 0


,user_id,has_purchase_last_year,has_purchase_last_qtr


user_segment
new_or_inactive    91582
active             10853
Name: count, dtype: int64
user_segment
new_or_inactive    89.4%
active             10.6%
Name: proportion, dtype: str
(102435, 4)
user_segment
new_or_inactive    91582
active             10853
Name: count, dtype: int64


,user_id,has_purchase_last_year,has_purchase_last_qtr,user_segment
0,9d415f62-5d38-436d-a443-336b2b759568,False,False,new_or_inactive
1,a71251a4-f215-4046-85fb-51206780e6c8,True,True,active
2,a8e69945-e3f7-4458-8ea9-883062d75c71,False,False,new_or_inactive
3,d2280282-85c3-40fe-93b8-990479b73fb3,False,False,new_or_inactive
4,19755aa9-50fc-44f2-80c0-e2eb4ecb4160,True,True,active
5,b632f58c-f467-4363-9b87-b01c237df412,False,False,new_or_inactive
6,ff30683f-bd0b-4ada-9cc8-547049fdc62c,False,False,new_or_inactive
7,3cc6932b-7cc1-4852-b6a8-3348607240eb,False,False,new_or_inactive
8,2d268310-e659-49d4-82b1-b18f344587d1,True,True,active
9,5c135263-b6ee-4cec-b431-82e90a062983,False,False,new_or_inactive


saved
